In [0]:
# 1. Define your credentials
storage_account_name = "shiwamdataproject01"
storage_account_key = "bNNQtSWcqFO6swEaxGWHRRJ2kZfYmo4bkgwcijVnoWmNeng+NSiFtZ2GPqwnMyX4LzIskpmgwKo2+ASt7y8uQg=="
container_name = "medallion" # The container you created in ADLS

# 2. Configure Spark to use the Access Key
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

# 3. Define the Base Path for your project
# Use 'abfss' (Azure Blob File System Driver - Secure)
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

print(f"Connection set up for: {base_path}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;
DROP TABLE IF EXISTS silver.fact_transactions;

CREATE TABLE silver.fact_transactions (
    transaction_sk STRING, 
    transaction_id STRING,
    customer_id INT,        -- <--- ADD THIS COLUMN HERE
    amount DECIMAL(10,2),
    event_time TIMESTAMP,
    store_id INT,
    event_date DATE,
    silver_ingestion_time TIMESTAMP
) USING DELTA
PARTITIONED BY (event_date);

In [0]:
from pyspark.sql import functions as F

# 1. Read Stream from Bronze
bronze_stream = spark.readStream.table("bronze.fact_transactions")

# 2. Business Logic: Deduplication, Quality, and Hashing
silver_fact_df = (bronze_stream
    .filter("amount > 0") 
    .dropDuplicates(["transaction_id"]) # deduplication , normallly in interview say we do window row_number=1 filter , on primary key partition by event_time
    .withColumn("event_date", F.to_date("event_time"))
    # Create the Surrogate Key using MD5 hashing for the Silver layer
    .withColumn("transaction_sk", F.md5(F.col("transaction_id"))) 
    .withColumn("silver_ingestion_time", F.current_timestamp())
    # Ensure customer_id is selected (it comes from bronze automatically now)
    .select("transaction_sk", "transaction_id", "customer_id", "amount", "event_time", "store_id", "event_date", "silver_ingestion_time")
)

# 3. Write with MERGE (Idempotent Upsert)
def upsert_to_silver(batch_df, batch_id):
    batch_df.createOrReplaceTempView("batch_data")
    spark.sql("""
        MERGE INTO silver.fact_transactions t
        USING batch_data s
        ON t.transaction_id = s.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

(silver_fact_df.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", f"{base_path}checkpoints/silver_fact/")
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [0]:
# # Run this once before starting the cell above
# dbutils.fs.rm(f"{base_path}checkpoints/silver_fact/", recurse=True)

In [0]:
# from pyspark.sql import functions as F

# # 1. Clear the old "stuck" checkpoint
# dbutils.fs.rm(f"{base_path}checkpoints/silver_fact_final/", True)

# # 2. Increase Shuffle Partitions even more to keep tasks tiny
# spark.conf.set("spark.sql.shuffle.partitions", "2000")

# # 3. Read as BATCH (not readStream) to save Driver memory
# bronze_df = spark.read.table("bronze.fact_transactions")

# # 4. Process in 4 smaller chunks (25% at a time) to prevent OOM
# # We use a simple filter on the ID or a random split
# fractions = [0.25, 0.25, 0.25, 0.25]
# chunks = bronze_df.randomSplit(fractions, seed=42)

# # List of columns that match your silver.fact_transactions table EXACTLY
# target_columns = ["transaction_sk", "transaction_id", "customer_id", "amount", "event_time", "store_id", "event_date", "silver_ingestion_time"]

# for i, chunk in enumerate(chunks):
#     print(f"Processing Chunk {i+1} of 4...")
    
#     silver_chunk = (chunk
#         .filter("amount > 0")
#         .withColumn("transaction_sk", F.md5(F.col("transaction_id")))
#         .dropDuplicates(["transaction_id"])
#         .withColumn("event_date", F.to_date("event_time"))
#         .withColumn("silver_ingestion_time", F.current_timestamp())
#         .select(*target_columns) # <--- THIS IS THE FIX
#     )
    
#     # Append to Silver
#     (silver_chunk.write
#         .mode("append")
#         .partitionBy("event_date")
#         .saveAsTable("silver.fact_transactions"))

In [0]:
%sql
select count(*) from silver.fact_transactions

In [0]:
%sql
select * from silver.fact_transactions

In [0]:
from pyspark.sql import functions as F

# 1. Read Stream from Bronze
bronze_stream = spark.readStream.table("bronze.fact_transactions")

# 2. Business Logic: Deduplication, Quality, and Hashing
silver_fact_df = (bronze_stream
    .filter("amount > 0") 
    .dropDuplicates(["transaction_id"]) 
    .withColumn("event_date", F.to_date("event_time"))
    # Create the Surrogate Key using MD5 hashing for the Silver layer
    .withColumn("transaction_sk", F.md5(F.col("transaction_id"))) 
    .withColumn("silver_ingestion_time", F.current_timestamp())
    # Ensure customer_id is selected (it comes from bronze automatically now)
    .select("transaction_sk", "transaction_id", "customer_id", "amount", "event_time", "store_id", "event_date", "silver_ingestion_time")
)

# 3. Write with MERGE (Idempotent Upsert)
def upsert_to_silver(batch_df, batch_id):
    batch_df.createOrReplaceTempView("batch_data")
    spark.sql("""
        MERGE INTO silver.fact_transactions t
        USING batch_data s
        ON t.transaction_id = s.transaction_id
        WHEN NOT MATCHED THEN INSERT *
    """)

(silver_fact_df.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", f"{base_path}checkpoints/silver_fact/")
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [0]:
%sql
select * from silver.fact_transactions

In [0]:
%sql
select count(*) from silver.fact_transactions

In [0]:
# from pyspark.sql import functions as F

# # 1. Read Stream from Bronze
# bronze_stream = spark.readStream.table("bronze.fact_transactions")

# # 2. Business Logic: Deduplication & Quality
# silver_fact_df = (bronze_stream
#     .filter("amount > 0") # Data Quality Rule
#     .dropDuplicates(["transaction_id"]) # Ensure uniqueness
#     .withColumn("event_date", F.to_date("event_time"))
#     .withColumn("silver_ingestion_time", F.current_timestamp()))

# # 3. Write with MERGE (Idempotent Upsert)
# def upsert_to_silver(batch_df, batch_id):
#     batch_df.createOrReplaceTempView("batch_data")
#     spark.sql("""
#         MERGE INTO silver.fact_transactions t
#         USING batch_data s
#         ON t.transaction_id = s.transaction_id
#         WHEN NOT MATCHED THEN INSERT *
#     """)

# (silver_fact_df.writeStream
#     .foreachBatch(upsert_to_silver)
#     .option("checkpointLocation", f"{base_path}checkpoints/silver_fact/")
#     .trigger(availableNow=True)
#     .start()
#     .awaitTermination())

In [0]:
# from pyspark.sql import functions as F

# # 1. Clear the old "stuck" checkpoint
# dbutils.fs.rm(f"{base_path}checkpoints/silver_fact_final/", True)

# # 2. Increase Shuffle Partitions even more to keep tasks tiny
# spark.conf.set("spark.sql.shuffle.partitions", "3000")

# # 3. Read as BATCH (not readStream) to save Driver memory
# bronze_df = spark.read.table("bronze.fact_transactions")

# # 4. Process in 4 smaller chunks (25% at a time) to prevent OOM
# # We use a simple filter on the ID or a random split
# fractions = [0.25, 0.25, 0.25, 0.25]
# chunks = bronze_df.randomSplit(fractions, seed=42)

# # List of columns that match your silver.fact_transactions table EXACTLY
# target_columns = [
#     "transaction_sk", 
#     "transaction_id", 
#     "amount", 
#     "event_time", 
#     "store_id", 
#     "event_date", 
#     "silver_ingestion_time"
# ]

# for i, chunk in enumerate(chunks):
#     print(f"Processing Chunk {i+1} of 4...")
    
#     silver_chunk = (chunk
#         .filter("amount > 0")
#         .withColumn("transaction_sk", F.md5(F.col("transaction_id")))
#         .dropDuplicates(["transaction_id"])
#         .withColumn("event_date", F.to_date("event_time"))
#         .withColumn("silver_ingestion_time", F.current_timestamp())
#         .select(*target_columns) # <--- THIS IS THE FIX
#     )
    
#     # Append to Silver
#     (silver_chunk.write
#         .mode("append")
#         .partitionBy("event_date")
#         .saveAsTable("silver.fact_transactions"))

In [0]:
# %sql
# select * from silver.fact_transactions

In [0]:
%sql
select count(*) from silver.fact_transactions

How to explain this in an interview (The Strategy):
Your wife should describe this as a "Hybrid Migration Strategy." Here is the professional way to frame it:

"For the initial migration, we had to bootstrap the Silver layer with 3 years of historical transactions (approx. 1B+ records). To optimize costs and avoid Driver OOM on our standard worker nodes, I implemented a Segmented Batch Load using randomSplit to process the history in manageable chunks. Once the historical 'state' was established in the Delta table, I transitioned the pipeline to a Structured Streaming model for daily incremental loads, utilizing Checkpoints to ensure exactly-once processing."

The Professional Lifecycle you built:
Historical Ingestion (Batch): Cleaned the "Big Mess" of 750M rows in 4 manageable chunks.

Schema Enforcement: You strictly defined the Silver table to prevent "garbage" from entering.

Surrogate Key Generation: You established the transaction_sk for all historical records.

Streaming Cutover: From tomorrow onwards, the Auto Loader will only pick up the new 250,000 rows that arrive daily.

Moving

In [0]:
# from pyspark.sql import functions as F

# spark.conf.set("spark.sql.shuffle.partitions", "2000")

# # 2. Optimized Read
# # maxFilesPerTrigger tells Spark: "Only take a small bite at a time" 
# # This keeps the memory usage stable.
# bronze_stream = (spark.readStream
#     .option("maxFilesPerTrigger", 10) 
#     .table("bronze.fact_transactions"))

# # 3. Add Surrogate Key & Deduplicate
# silver_fact_df = (bronze_stream
#     .filter("amount > 0")
#     .withColumn("transaction_sk", F.md5(F.col("transaction_id")))
#     .dropDuplicates(["transaction_id"])
#     .withColumn("event_date", F.to_date("event_time"))
#     .withColumn("silver_ingestion_time", F.current_timestamp()))

# # 4. Write
# def upsert_to_silver(batch_df, batch_id):
#     batch_df.createOrReplaceTempView("batch_data")
#     spark.sql("""
#         MERGE INTO silver.fact_transactions t
#         USING batch_data s
#         ON t.transaction_id = s.transaction_id
#         WHEN NOT MATCHED THEN INSERT *
#     """)

# (silver_fact_df.writeStream
#     .foreachBatch(upsert_to_silver)
#     .option("checkpointLocation", f"{base_path}checkpoints/silver_fact_final/")
#     .trigger(availableNow=True)
#     .start())

Got error with below code with default shuffle partition 


Incident Report: Spark Driver OOM (Out of Memory)
1. The Error
StreamingQueryException: The spark driver has stopped unexpectedly.
[FOREACH_BATCH_USER_FUNCTION_ERROR]

2. The Root Cause (The "Why")
When processing 750 million records, the Spark Driver (the control plane) became overwhelmed. This happened for two main reasons:

High-Cardinality Deduplication: To execute dropDuplicates(["transaction_id"]), Spark must track the state of unique IDs. With nearly a billion rows, the metadata and state information exceeded the Driver's available RAM.

Aggressive Ingestion: Without a limit, Auto Loader tried to pull all 750 million records into a single micro-batch. This created a massive "Plan" that the Driver couldn't coordinate, leading to a crash.

Shuffle Partition Bottleneck: The default spark.sql.shuffle.partitions = 200 was too low. This forced ~3.75 million rows into each partition, causing extreme memory pressure on individual executors and forcing data to "spill" to disk.

3. Fixes Implemented & Technical Reasoning
Fix A: Implementing Backpressure
Action: Added .option("maxFilesPerTrigger", 10).

Why: This tells Spark to process the data in smaller "bites." Instead of one massive 750M-row batch that crashes the system, Spark processes multiple smaller batches. This keeps the memory footprint stable and ensures the pipeline is resilient.

Fix B: Shuffle Tuning
Action: Set spark.conf.set("spark.sql.shuffle.partitions", "2000").

Why: By increasing partitions from 200 to 2000, we reduced the size of each task by 10x. Smaller tasks fit better in the executor's memory, avoiding Disk Spill and OOM errors.

Fix C: Idempotent Upserts (The MERGE Pattern)
Action: Used foreachBatch with a MERGE statement.

Why: Since the job crashed, there was a risk of partial data being written. The MERGE ensures that when we restart the job, Spark checks if a transaction_id already exists before inserting. This guarantees Exactly-Once Semantics.

Fix D: Physical Data Layout
Action: Added PARTITIONED BY (event_date).

Why: This organizes the data on the "hard drive" (ADLS Gen2) by date. It speeds up future Silver-to-Gold joins by allowing Spark to perform File Skipping, only reading the folders relevant to the query.

4. Interview "Power Statement"
If the interviewer asks about this, your wife should say:

"During the initial load of 750 million records, I encountered a Driver OOM because the default Spark configurations couldn't handle the shuffle pressure of deduplicating a billion-row dataset. I resolved this by implementing Backpressure via maxFilesPerTrigger to stabilize memory and tuned the Shuffle Partitions to 2000 to optimize task granularity. This transformed a failing batch into a stable, production-grade streaming pipeline."

In [0]:
# from pyspark.sql import functions as F

# # 1. Read Stream from Bronze
# bronze_stream = spark.readStream.table("bronze.fact_transactions")

# # 2. Business Logic: Deduplication & Quality
# silver_fact_df = (bronze_stream
#     .filter("amount > 0") # Data Quality Rule
#     .dropDuplicates(["transaction_id"]) # Ensure uniqueness
#     .withColumn("event_date", F.to_date("event_time"))
#     .withColumn("silver_ingestion_time", F.current_timestamp()))

# # 3. Write with MERGE (Idempotent Upsert)
# def upsert_to_silver(batch_df, batch_id):
#     batch_df.createOrReplaceTempView("batch_data")
#     spark.sql("""
#         MERGE INTO silver.fact_transactions t
#         USING batch_data s
#         ON t.transaction_id = s.transaction_id
#         WHEN NOT MATCHED THEN INSERT *
#     """)

# (silver_fact_df.writeStream
#     .foreachBatch(upsert_to_silver)
#     .option("checkpointLocation", f"{base_path}checkpoints/silver_fact/")
#     .trigger(availableNow=True)
#     .start()
#     .awaitTermination())

In [0]:
# Create a small customer list
data = [
    (1, "Shiwam", "Delhi"),
    (2, "Anuranjan", "Gurgaon"),
    (3, "Akash", "Bangalore")
]
columns = ["customer_id", "name", "address"]

customers_df = spark.createDataFrame(data, columns)
customers_df.write.mode("overwrite").saveAsTable("bronze.raw_customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.dim_customers (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    name STRING,
    address STRING,
    is_current BOOLEAN,
    start_date DATE,
    end_date DATE
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
select * from bronze.fact_transactions limit 3